<center><h1>Case_John_HW3</h1></center>
<br>
<br>

Name: John Case
<br>
Github Username: johncase17
<br>
USC ID: 2249212508

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [175]:
from ensurepip import bootstrap

# %pip install -r ../requitements.txt

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path
from scipy.stats import bootstrap
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

Get the AReM Data Set

In [176]:
#skip 4 rows to extract columns, 5 for data rows only
def extract_data(filepath):
    return pd.read_csv(filepath, sep=r"[,\s]+", comment='#', engine = "python", header = None)
test = extract_data("../data/AReM/standing/dataset13.csv")

# Columns: time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
def append_data(filepath, df):
    return pd.concat([df, extract_data(filepath)], ignore_index=True)
print(test.head())
print(append_data("../data/AReM/standing/dataset14.csv", test))

      0     1    2      3     4      5     6
0     0  45.0  0.0  12.50  0.87  14.50  0.50
1   250  45.0  0.0  13.00  0.00  14.00  1.00
2   500  45.0  0.0  12.00  0.00  13.50  0.87
3   750  45.0  0.0  13.33  0.47  12.50  0.50
4  1000  45.0  0.0  12.25  0.43  12.75  0.83
          0      1     2      3     4      5     6
0         0  45.00  0.00  12.50  0.87  14.50  0.50
1       250  45.00  0.00  13.00  0.00  14.00  1.00
2       500  45.00  0.00  12.00  0.00  13.50  0.87
3       750  45.00  0.00  13.33  0.47  12.50  0.50
4      1000  45.00  0.00  12.25  0.43  12.75  0.83
5      1250  45.00  0.00  12.33  0.47  12.50  0.50
6      1500  45.00  0.00  12.75  0.43  12.50  0.50
7      1750  45.00  0.00  13.25  0.43  12.50  0.87
8      2000  45.00  0.00  12.00  0.00  12.50  0.50
9      2250  45.00  0.00  12.25  0.43  13.00  1.00
10     2500  45.00  0.00  12.50  0.50  13.50  1.50
11     2750  45.00  0.00  10.75  1.09  12.50  0.50
12     3000  45.00  0.00   9.67  0.94  12.67  0.94
13     3250  45.

### (b) Test and Train Data

In [177]:
path1 = "../data/AReM/bending1"
path2 = "../data/AReM/bending2"
path3 = "../data/AReM/cycling"
path4 = "../data/AReM/lying"
path5 = "../data/AReM/sitting"
path6 = "../data/AReM/standing"
path7 = "../data/AReM/walking"
bend_paths = [path1, path2]
other_paths = [path3, path4, path5, path6, path7]
testing_data = [[pd.DataFrame()] * 2, [pd.DataFrame()] * 2, [pd.DataFrame()] * 3, [pd.DataFrame()] * 3, [pd.DataFrame()] * 3, [pd.DataFrame()] * 3, [pd.DataFrame()] * 3]
training_data = [[pd.DataFrame()] * 5, [pd.DataFrame()] * 4, [pd.DataFrame()] * 12, [pd.DataFrame()] * 12, [pd.DataFrame()] * 12, [pd.DataFrame()] * 12, [pd.DataFrame()] * 12]
data_index = 0
#bending loop: train on datasets 1 and 2, test on others
#assumes folders are properly labeled
for folder in bend_paths:
    loc = Path(folder)
    for file in loc.glob("*.csv"):
        i = int(file.stem.removeprefix("dataset")) - 1
        print(file.name)
        if i < 2:
            testing_data[data_index][i] = extract_data(file)
        else:
            training_data[data_index][i - 2] = extract_data(file)
    data_index = data_index + 1


#other datasets: train on datasets 1, 2, 3, test on others
#assumes folders are properly labeled
for folder in other_paths:
    loc = Path(folder)
    for file in loc.glob("*.csv"):
        i = int(file.stem.removeprefix("dataset")) - 1
        print(str(i) + str(data_index) + file.name)
        if i < 3:
            testing_data[data_index][i] = extract_data(file)
        else:
            training_data[data_index][i - 3] =extract_data(file)
    data_index = data_index + 1


dataset1.csv
dataset2.csv
dataset3.csv
dataset4.csv
dataset5.csv
dataset6.csv
dataset7.csv
dataset1.csv
dataset2.csv
dataset3.csv
dataset4.csv
dataset5.csv
dataset6.csv
02dataset1.csv
92dataset10.csv
102dataset11.csv
112dataset12.csv
122dataset13.csv
132dataset14.csv
142dataset15.csv
12dataset2.csv
22dataset3.csv
32dataset4.csv
42dataset5.csv
52dataset6.csv
62dataset7.csv
72dataset8.csv
82dataset9.csv
03dataset1.csv
93dataset10.csv
103dataset11.csv
113dataset12.csv
123dataset13.csv
133dataset14.csv
143dataset15.csv
13dataset2.csv
23dataset3.csv
33dataset4.csv
43dataset5.csv
53dataset6.csv
63dataset7.csv
73dataset8.csv
83dataset9.csv
04dataset1.csv
94dataset10.csv
104dataset11.csv
114dataset12.csv
124dataset13.csv
134dataset14.csv
144dataset15.csv
14dataset2.csv
24dataset3.csv
34dataset4.csv
44dataset5.csv
54dataset6.csv
64dataset7.csv
74dataset8.csv
84dataset9.csv
05dataset1.csv
95dataset10.csv
105dataset11.csv
115dataset12.csv
125dataset13.csv
135dataset14.csv
145dataset15.csv
15datas

### (c) Feature Extraction

#### i. Research

Common statistical features: Mean, variance, min, max, 1st quartile, 2nd quartile (median), 3rd quartile

<B></B>

Time-specific feature: autocorrelation

#### ii. Extraction

In [178]:
#flatten + test
flatened_data = [dframe for list in (testing_data + training_data) for dframe in list]
print(len(flatened_data))
print(flatened_data[0].head())

88
      0      1     2      3     4      5    6
0     0  39.25  0.43  22.75  0.43  33.75  1.3
1   250  39.25  0.43  23.00  0.00  33.00  0.0
2   500  39.25  0.43  23.25  0.43  33.00  0.0
3   750  39.50  0.50  23.00  0.71  33.00  0.0
4  1000  39.50  0.50  24.00  0.00  33.00  0.0


In [179]:
feature_names = ['avg_rss12','var_rss12','avg_rss13','var_rss13','avg_rss23','var_rss23']
output_columns = ['Instance', 'min1', 'max1', 'mean1', 'median1', 'SD1', '1st_quart1', '3rd_quart1', 'min2', 'max2', 'mean2', 'median2', 'SD2', '1st_quart2', '3rd_quart2', 'min3', 'max3', 'mean3', 'median3', 'SD3', '1st_quart3', '3rd_quart3', 'min4', 'max4', 'mean4', 'median4', 'SD4', '1st_quart4', '3rd_quart4', 'min5', 'max5', 'mean5', 'median5', 'SD5', '1st_quart5', '3rd_quart5', 'min6', 'max6', 'mean6', 'median6', 'SD6', '1st_quart6', '3rd_quart6']
output_dataset = pd.DataFrame(np.nan, index=range(len(flatened_data)), columns=output_columns)

#index is the dataframe corresponding to a single dataset
#i tracks which input feature is being processed
#j tracks which time series feature is being used
# 0=min, 1=max, 2=mean, 3= median, 4=SD, 5=1st_quart, 6=3rd_quart
index = 0
while index < 88:
    output_dataset.iloc[index, 0] = index + 1
    for i in range (1,7):
        for j in range (0,7):
            col_num = 1+(i-1)*7+j
            if j == 0:
                output_dataset.iloc[index, col_num] = flatened_data[index][i].min()
            if j == 1:
                output_dataset.iloc[index, col_num] = flatened_data[index][i].max()
            if j == 2:
                output_dataset.iloc[index, col_num] = flatened_data[index][i].mean()
            if j == 3:
                output_dataset.iloc[index, col_num] = flatened_data[index][i].median()
            if j == 4:
                output_dataset.iloc[index, col_num] = flatened_data[index][i].std()
            if j == 5:
                output_dataset.iloc[index, col_num] = flatened_data[index][i].quantile(0.25)
            if j == 6:
                output_dataset.iloc[index, col_num] = flatened_data[index][i].quantile(0.75)
        col_num = col_num + 1
    # feature = feature + 1
    index = index + 1
print(output_dataset)

    Instance   min1   max1      mean1  median1       SD1  1st_quart1  \
0        1.0  37.25  45.00  40.624792   40.500  1.476967     39.2500   
1        2.0  38.00  45.67  42.812812   42.500  1.435550     42.0000   
2        3.0  12.75  51.00  24.562958   24.250  3.737514     23.1875   
3        4.0   0.00  42.75  27.464604   28.000  3.583582     25.5000   
4        5.0  24.25  45.00  37.177042   36.250  3.581301     34.5000   
5        6.0  28.75  44.75  37.561187   36.875  3.226507     35.2500   
6        7.0  22.00  44.67  37.058708   36.000  3.710180     34.5000   
7        8.0  23.50  30.00  27.716375   27.500  1.442253     27.0000   
8        9.0  24.75  48.33  44.182937   48.000  7.495615     48.0000   
9       10.0  48.00  48.25  48.004167   48.000  0.032038     48.0000   
10      11.0  33.25  48.00  42.363563   43.000  2.068247     42.0000   
11      12.0  37.00  50.75  45.917667   45.635  1.089027     45.0000   
12      13.0  44.50  46.75  45.239667   45.000  0.429915     45.

#### iii. Standard Deviation

In [180]:
#std for each feature
features = output_dataset.columns.drop('Instance')
std_interval = pd.DataFrame(dtype=object, index=range(len(features)), columns=['feature', 'std', 'lower', 'upper'])
index = 0
for feature in features:
    data = output_dataset[feature].to_numpy()
    std = bootstrap(data=(data,), confidence_level=0.9, statistic=np.std, random_state=42, method='percentile')
    std_interval.loc[index, 'lower'] = std.confidence_interval.low
    std_interval.loc[index, 'upper'] = std.confidence_interval.high
    std_interval.loc[index, 'feature'] = feature
    std_interval.loc[index, 'std'] = output_dataset[feature].std(ddof=0)
    index = index + 1

print(std_interval)



       feature       std     lower      upper
0         min1  9.515445  8.200262  10.691329
1         max1  4.369322  3.338077   5.236803
2        mean1  5.305314  4.683571   5.855863
3      median1  5.409056  4.764524   5.971714
4          SD1  1.762056   1.56142   1.940672
5   1st_quart1  6.118526  5.542267   6.612373
6   3rd_quart1  5.109643  4.318389   5.811497
7         min2       0.0       0.0        0.0
8         max2  5.033882  4.597552    5.37516
9        mean2  1.565194  1.387118    1.69692
10     median2  1.404197  1.230651     1.5343
11         SD2  0.879068  0.798673   0.937313
12  1st_quart2  0.940994   0.82605   1.028968
13  3rd_quart2  2.113157  1.880897    2.28237
14        min3  2.939616  2.740544   3.090272
15        max3  4.847358  4.156121    5.43123
16       mean3   3.98554  3.390391   4.462973
17     median3  4.013397  3.406623   4.508701
18         SD3  0.941316   0.75766   1.119716
19  1st_quart3  4.196608  3.601375    4.66712
20  3rd_quart3  4.147858  3.523439

#### iv. Select Features

The time-series features I choose to select are mean, standard deviation, and max.
<br>
median is somewhat redundant, and min is less important: in some datasets it's all identical values.

## 2. ISLR 3.7.4

### (a) Linear Train

The training RSS for the cubic model will be lower or the same, as it can select the best-possible linear model by setting coefficients of higher-ordered powers to 0 if ideal, or set them to different values if it yields lower training RSS (which is likely).

### (b) Linear Test

I would expect the test RSS to be lower for the linear model, as it does not have bias, while the cubic model likely has higher order coefficients increasing variance.

### (c) Not Linear Train

The cubic model should still have a lower or equal training RSS because if a linear approximation minimizes RSS, the cubic model can recreate the same formula as the linear model. Also, because the linear model cannot capture a non-linear relationship, the higher-order model is able to more closely approximate it.

### (d) Not Linear Testing

In terms of test RSS, it's unclear, as if the relationship is closer to linear than cubic, the cubic model's variance will result in greater error, whereas if the relationship is less linear, the cubic model will better fit the relationship.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 